# Sliding-window attention: checkpoint inference benchmark

This notebook loads one trained dense `AdaptiveESM` checkpoint into both the original model and the checkpoint-compatible sliding-window model. It compares **incremental peak inference memory** (CUDA peak allocation above resident model/input memory) and synchronized **end-to-end inference latency** across window sizes and measured MLP/QK width-cap counterfactuals.

The local CUDA path uses a cached FlexAttention `BlockMask` and a compiled kernel, following the [PyTorch FlexAttention blog](https://pytorch.org/blog/flexattention/) and [PyTorch 2.13 documentation](https://docs.pytorch.org/docs/2.13/nn.attention.flex_attention.html). Compilation and warm-up are intentionally excluded from the timed measurements.

In [ ]:
from __future__ import annotations

import gc
import json
import os
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from distribution import inv_softplus

from esm_surrogate.model import AdaptiveESM as DenseAdaptiveESM
from esm_surrogate.model_sliding_attention import (
    AdaptiveESM as SlidingWindowAdaptiveESM,
    clear_sliding_window_mask_cache,
)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_ROOT = Path(
    os.environ.get(
        "ESM_SURROGATE_RESULTS_ROOT",
        str(PROJECT_ROOT / "RESULTS"),
    )
)
checkpoint_override = os.environ.get("ESM_SURROGATE_CHECKPOINT")
if checkpoint_override:
    CHECKPOINT_PATH = Path(checkpoint_override).expanduser().resolve()
else:
    candidates = sorted(
        RESULTS_ROOT.glob(
            "esm_distillation_*_ProteinEmbeddingDataset/**/best_checkpoint.pth"
        )
    )
    if not candidates:
        raise FileNotFoundError(
            "No ESM checkpoint found. Set ESM_SURROGATE_CHECKPOINT explicitly."
        )
    CHECKPOINT_PATH = candidates[0]

MANIFEST_PATH = CHECKPOINT_PATH.with_name("model_manifest.json")


def select_benchmark_device() -> tuple[torch.device, str]:
    """Prefer the least-loaded visible GPU unless the user selects one."""
    explicit_device = os.environ.get("ESM_SURROGATE_BENCHMARK_DEVICE")
    if explicit_device:
        return torch.device(explicit_device), "explicit environment override"

    try:
        query = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=index,uuid,utilization.gpu,memory.used,memory.total",
                "--format=csv,noheader,nounits",
            ],
            check=True,
            capture_output=True,
            text=True,
        )
        physical_gpus = []
        for line in query.stdout.splitlines():
            physical_index, uuid, utilization, memory_used, memory_total = (
                value.strip() for value in line.split(",")
            )
            physical_gpus.append(
                {
                    "physical_index": physical_index,
                    "uuid": uuid,
                    "utilization": int(utilization),
                    "memory_used": int(memory_used),
                    "memory_total": int(memory_total),
                }
            )

        # Map CUDA's logical devices back to nvidia-smi UUIDs. This remains
        # correct when CUDA_VISIBLE_DEVICES filters or reorders the GPUs.
        candidates = []
        for logical_index in range(torch.cuda.device_count()):
            visible_uuid = str(torch.cuda.get_device_properties(logical_index).uuid)
            matching = next(
                (
                    gpu
                    for gpu in physical_gpus
                    if gpu["uuid"].removeprefix("GPU-") == visible_uuid
                ),
                None,
            )
            if matching is not None:
                candidates.append({**matching, "logical_index": logical_index})

        if not candidates:
            raise RuntimeError("No visible NVIDIA GPUs were reported by nvidia-smi.")
        selected = min(
            candidates,
            key=lambda gpu: (
                gpu["utilization"],
                gpu["memory_used"] / max(gpu["memory_total"], 1),
                gpu["logical_index"],
            ),
        )
        description = (
            f"auto-selected logical cuda:{selected['logical_index']} / physical "
            f"GPU {selected['physical_index']} at {selected['utilization']}% utilization, "
            f"{selected['memory_used']} MiB used"
        )
        return torch.device(f"cuda:{selected['logical_index']}"), description
    except (FileNotFoundError, subprocess.SubprocessError, ValueError) as error:
        return torch.device("cuda"), f"automatic selection unavailable: {error}"


DEVICE, DEVICE_SELECTION = select_benchmark_device()
SEQUENCE_LENGTH = int(os.environ.get("ESM_SURROGATE_SEQUENCE_LENGTH", "1300"))
WINDOW_RADII = [
    int(value)
    for value in os.environ.get("ESM_SURROGATE_WINDOW_RADII", "16,32,64,128,256").split(",")
]
COUNTERFACTUAL_RADIUS = int(
    os.environ.get("ESM_SURROGATE_COUNTERFACTUAL_RADIUS", "64")
)
MLP_WIDTH_CAPS = [
    int(value)
    for value in os.environ.get("ESM_SURROGATE_MLP_WIDTH_CAPS", "300,256,128,64,32").split(",")
]
QK_HEAD_WIDTH_CAPS = [
    int(value)
    for value in os.environ.get("ESM_SURROGATE_QK_HEAD_WIDTH_CAPS", "300,256,128,64,32").split(",")
]
BLOCK_SIZE = int(os.environ.get("ESM_SURROGATE_BLOCK_SIZE", "64"))
WARMUP_RUNS = int(os.environ.get("ESM_SURROGATE_WARMUP_RUNS", "2"))
TIMED_RUNS = int(os.environ.get("ESM_SURROGATE_TIMED_RUNS", "10"))
USE_AUTOCAST = os.environ.get("ESM_SURROGATE_AUTOCAST", "1") != "0"
AMP_DTYPE = torch.bfloat16

if COUNTERFACTUAL_RADIUS not in WINDOW_RADII:
    raise ValueError("ESM_SURROGATE_COUNTERFACTUAL_RADIUS must occur in WINDOW_RADII.")
if DEVICE.type != "cuda" or not torch.cuda.is_available():
    raise RuntimeError("The efficient FlexAttention benchmark requires CUDA.")
if not CHECKPOINT_PATH.is_file() or not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"Expected checkpoint and model_manifest.json beside it: {CHECKPOINT_PATH}"
    )

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(DEVICE)} ({DEVICE_SELECTION})")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Sequence length: {SEQUENCE_LENGTH}; windows: {[2*r + 1 for r in WINDOW_RADII]}")
print(
    f"Counterfactual window: {2 * COUNTERFACTUAL_RADIUS + 1}; "
    f"MLP caps: {MLP_WIDTH_CAPS}; Q/K head caps: {QK_HEAD_WIDTH_CAPS}"
)

In [ ]:
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
model_config = dict(manifest["config"])
dim_input_features = int(manifest["dim_input_features"])
dim_target = int(manifest["dim_target"])

checkpoint_payload = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True,
)
checkpoint_state = checkpoint_payload.get(
    "model_state",
    checkpoint_payload.get("model_state_dict"),
)
if checkpoint_state is None:
    raise KeyError("Checkpoint has no model_state or model_state_dict entry.")

# Token identity does not affect allocation shapes. A deterministic, valid
# 1,300-residue protein gives the same production geometry as the cached data.
import esm

alphabet = esm.data.Alphabet.from_architecture("ESM-1b")
converter = alphabet.get_batch_converter(dim_input_features)
_, _, tokens = converter([("benchmark", "A" * SEQUENCE_LENGTH)])
tokens = tokens.to(DEVICE)
print(f"Token tensor: {tuple(tokens.shape)}")

In [ ]:
def release_cuda_memory() -> None:
    """Release unreferenced CUDA allocations between model variants."""
    gc.collect()
    torch.cuda.empty_cache()


def _set_distribution_width(distribution: torch.nn.Module, width: int) -> None:
    """Set one adaptive distribution to materialize an exact nested prefix."""
    base_distribution = distribution.discretized_distribution.base_distribution
    desired_rate = base_distribution._find_rate(
        float(width), distribution.truncation_quantile
    )
    with torch.no_grad():
        base_distribution._rate.copy_(
            inv_softplus(desired_rate.reshape_as(base_distribution._rate))
        )
    distribution.x = None
    actual_width = distribution.compute_truncation_number()
    if actual_width != width:
        raise RuntimeError(f"Requested width {width}, materialized {actual_width}.")


def apply_width_caps(
    model: SlidingWindowAdaptiveESM,
    *,
    mlp_width_cap: int | None = None,
    qk_head_width_cap: int | None = None,
) -> dict[str, list[int]]:
    """Truncate checkpoint-supported nested widths without inventing weights."""
    checkpoint_mlp_widths = model.instantiated_mlp_widths()
    checkpoint_qk_head_widths = model.instantiated_attention_head_widths()
    target_mlp_widths = [
        min(width, mlp_width_cap) if mlp_width_cap is not None else width
        for width in checkpoint_mlp_widths
    ]
    target_qk_head_widths = [
        min(width, qk_head_width_cap) if qk_head_width_cap is not None else width
        for width in checkpoint_qk_head_widths
    ]
    if min(target_mlp_widths + target_qk_head_widths) < 1:
        raise ValueError("All counterfactual width caps must be positive.")

    for distribution, width in zip(model.variational_widths, target_mlp_widths):
        _set_distribution_width(distribution, width)
    for distribution, width in zip(
        model.attention_variational_widths, target_qk_head_widths
    ):
        _set_distribution_width(distribution, width)
    model.change_width()
    return {
        "mlp_widths": model.instantiated_mlp_widths(),
        "qk_head_widths": model.instantiated_attention_head_widths(),
    }


def build_model(
    model_class: type[torch.nn.Module],
    radius: int | None = None,
    *,
    mlp_width_cap: int | None = None,
    qk_head_width_cap: int | None = None,
):
    """Restore the checkpoint and optionally truncate its nested widths."""
    config = dict(model_config)
    if radius is not None:
        config["attention_window_radius"] = radius
        config["attention_block_size"] = BLOCK_SIZE
    model = model_class(dim_input_features, dim_target, config)
    model.load_state_dict(checkpoint_state, strict=True)
    if mlp_width_cap is not None or qk_head_width_cap is not None:
        if not isinstance(model, SlidingWindowAdaptiveESM):
            raise TypeError("Width caps are only benchmarked on the sliding model.")
        apply_width_caps(
            model,
            mlp_width_cap=mlp_width_cap,
            qk_head_width_cap=qk_head_width_cap,
        )
    return model.eval().to(DEVICE)


def benchmark_model(model: torch.nn.Module) -> tuple[float, float, torch.Tensor]:
    """Return incremental peak MiB, median latency in ms, and one CPU output."""
    autocast = lambda: torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
        enabled=USE_AUTOCAST,
    )
    with torch.inference_mode():
        # FlexAttention compilation happens here and is excluded from results.
        for _ in range(WARMUP_RUNS):
            with autocast():
                model(tokens)
        torch.cuda.synchronize(DEVICE)

        gc.collect()
        torch.cuda.empty_cache()
        resident_bytes = torch.cuda.memory_allocated(DEVICE)
        torch.cuda.reset_peak_memory_stats(DEVICE)
        with autocast():
            output = model(tokens)[0]
        torch.cuda.synchronize(DEVICE)
        incremental_peak_mib = (
            torch.cuda.max_memory_allocated(DEVICE) - resident_bytes
        ) / (1024**2)

        elapsed_times = []
        for _ in range(TIMED_RUNS):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            with autocast():
                output = model(tokens)[0]
            end.record()
            end.synchronize()
            elapsed_times.append(start.elapsed_time(end))

    latency_ms = float(torch.tensor(elapsed_times).median())
    return incremental_peak_mib, latency_ms, output.float().cpu()

In [ ]:
results = []
dense_model = build_model(DenseAdaptiveESM)
dense_memory, dense_latency, dense_output = benchmark_model(dense_model)
results.append(
    {
        "label": "Original full",
        "window_size": None,
        "memory_mib": dense_memory,
        "latency_ms": dense_latency,
        "mean_abs_output_change": 0.0,
    }
)
del dense_model
release_cuda_memory()

sliding_outputs_by_radius = {}
sliding_model = build_model(SlidingWindowAdaptiveESM, radius=WINDOW_RADII[0])
checkpoint_mlp_widths = sliding_model.instantiated_mlp_widths()
checkpoint_qk_head_widths = sliding_model.instantiated_attention_head_widths()
for radius in WINDOW_RADII:
    sliding_model.set_attention_window_radius(radius)
    memory_mib, latency_ms, output = benchmark_model(sliding_model)
    sliding_outputs_by_radius[radius] = output
    results.append(
        {
            "label": f"Window {2 * radius + 1}",
            "window_size": 2 * radius + 1,
            "memory_mib": memory_mib,
            "latency_ms": latency_ms,
            "mean_abs_output_change": (output - dense_output).abs().mean().item(),
        }
    )
del sliding_model
release_cuda_memory()

print(f"{'Variant':<18} {'Peak MiB':>10} {'Median ms':>12} {'Mean |Δ output|':>17}")
for result in results:
    print(
        f"{result['label']:<18} {result['memory_mib']:>10.1f} "
        f"{result['latency_ms']:>12.2f} "
        f"{result['mean_abs_output_change']:>17.6f}"
    )

In [ ]:
# Measured structural counterfactuals at one representative local window.
# Caps only remove nested checkpoint channels; they never invent unavailable weights.
counterfactual_window_size = 2 * COUNTERFACTUAL_RADIUS + 1
counterfactual_reference_output = sliding_outputs_by_radius[COUNTERFACTUAL_RADIUS]
counterfactual_baseline = next(
    result
    for result in results
    if result["window_size"] == counterfactual_window_size
)
counterfactual_results = [
    {
        **counterfactual_baseline,
        "label": "Checkpoint widths",
        "mlp_widths": checkpoint_mlp_widths,
        "qk_head_widths": checkpoint_qk_head_widths,
        "mean_abs_change_from_window": 0.0,
    }
]

variant_specs = [
    *[
        {"label": f"MLP cap {cap}", "mlp_width_cap": cap}
        for cap in MLP_WIDTH_CAPS
    ],
    *[
        {"label": f"Q/K cap {cap}/head", "qk_head_width_cap": cap}
        for cap in QK_HEAD_WIDTH_CAPS
    ],
]

for spec in variant_specs:
    counterfactual_model = build_model(
        SlidingWindowAdaptiveESM,
        radius=COUNTERFACTUAL_RADIUS,
        mlp_width_cap=spec.get("mlp_width_cap"),
        qk_head_width_cap=spec.get("qk_head_width_cap"),
    )
    materialized_mlp_widths = counterfactual_model.instantiated_mlp_widths()
    materialized_qk_head_widths = (
        counterfactual_model.instantiated_attention_head_widths()
    )
    memory_mib, latency_ms, output = benchmark_model(counterfactual_model)
    counterfactual_results.append(
        {
            "label": spec["label"],
            "window_size": counterfactual_window_size,
            "memory_mib": memory_mib,
            "latency_ms": latency_ms,
            "mlp_widths": materialized_mlp_widths,
            "qk_head_widths": materialized_qk_head_widths,
            "mean_abs_change_from_window": (
                output - counterfactual_reference_output
            ).abs().mean().item(),
        }
    )
    del counterfactual_model
    release_cuda_memory()


def width_range(widths: list[int]) -> str:
    """Format the materialized width range across transformer layers."""
    return f"{min(widths)}-{max(widths)}"


print(
    f"\nCounterfactuals at window {counterfactual_window_size} "
    "(all figures are measured forwards)"
)
print(
    f"{'Variant':<22} {'MLP widths':>12} {'Q/K per head':>14} "
    f"{'Q/K projection':>15} {'Peak MiB':>10} {'Median ms':>12} "
    f"{'Mean |Δ window|':>17}"
)
for result in counterfactual_results:
    qk_projection_widths = [
        width * int(model_config["attention_heads"])
        for width in result["qk_head_widths"]
    ]
    print(
        f"{result['label']:<22} {width_range(result['mlp_widths']):>12} "
        f"{width_range(result['qk_head_widths']):>14} "
        f"{width_range(qk_projection_widths):>15} "
        f"{result['memory_mib']:>10.1f} {result['latency_ms']:>12.2f} "
        f"{result['mean_abs_change_from_window']:>17.6f}"
    )

In [ ]:
def draw_bars(ax, data, metric, colors, value_format):
    """Draw one consistently formatted benchmark panel."""
    bars = ax.bar(
        [result["label"] for result in data],
        [result[metric] for result in data],
        color=colors,
    )
    ax.bar_label(bars, fmt=value_format, padding=3, fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.2)
    ax.tick_params(axis="x", labelrotation=18)
    for label in ax.get_xticklabels():
        label.set_horizontalalignment("right")


# Plot 1: window sweep and measured width-counterfactual memory.
window_colors = ["#4477AA"] + ["#228833"] * (len(results) - 1)
width_colors = ["#228833"] + ["#CC6677"] * len(MLP_WIDTH_CAPS) + [
    "#AA4499"
] * len(QK_HEAD_WIDTH_CAPS)
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
draw_bars(axes[0], results, "memory_mib", window_colors, "%.0f MiB")
draw_bars(
    axes[1], counterfactual_results, "memory_mib", width_colors, "%.0f MiB"
)
axes[0].set_ylabel("Incremental peak CUDA memory (MiB)")
axes[0].set_title(f"Window sweep at {tokens.size(1)} tokens")
axes[1].set_title(f"Width caps with window {counterfactual_window_size}")
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: synchronized end-to-end checkpoint inference latency.
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
draw_bars(axes[0], results, "latency_ms", window_colors, "%.1f ms")
draw_bars(
    axes[1], counterfactual_results, "latency_ms", width_colors, "%.1f ms"
)
axes[0].set_ylabel("Median inference time (ms)")
axes[0].set_title(f"Window sweep ({TIMED_RUNS} timed runs)")
axes[1].set_title(
    f"Width caps at window {counterfactual_window_size} ({TIMED_RUNS} timed runs)"
)
plt.tight_layout()
plt.show()

## Reading the comparison

- Memory excludes resident checkpoint parameters and inputs, isolating peak allocations caused by the forward pass.
- Latency covers the complete model forward, including adaptive-width bookkeeping, projections, transformer layers, and pooling. FlexAttention compilation is excluded.
- `Mean |Δ output|` is reported because a finite local window intentionally changes the checkpoint computation; speed and memory should be evaluated alongside downstream accuracy before adopting a window.
- Width counterfactuals use the same checkpoint and representative local window. A cap keeps `min(checkpoint width, cap)` in every layer, so it only removes trained nested channels and never invents weights absent from the checkpoint.
- MLP caps are hidden dimensions. Q/K caps are dimensions per attention head; the table also reports the corresponding total projection width after multiplying by the configured number of heads.
- `Mean |Δ window|` isolates the effect of each width cap relative to the unchanged checkpoint-width model at the same local window. These structural measurements do not establish downstream accuracy.